# 04 — LegalBert-pt + LoRA Ponderado (opcional, requer GPU)

Fine-tuning do `dominguesm/legal-bert-base-cased-ptbr` com **LoRA** sobre
**`VOTO_LIMPO`**, com pesos de classe (cross-entropy ponderada ou Focal Loss).
Mesmos guardrails dos notebooks anteriores: nenhuma feature vem do SUMARIO,
e o corpus já passou pelo gate `auditar_vazamento` no notebook 01.

**Requer GPU** (Colab: Runtime → Alterar tipo de ambiente → T4). Sem GPU, a
célula de configuração reduz drasticamente o escopo (menos folds, menos
épocas, sequência mais curta) só para **validar o fluxo** — os números dessa
execução reduzida não são resultado final.

**Custo esperado (com T4):** ~15–20 min por fold × 5 folds ≈ 1,5–2h para
`n_splits=5`. Ajuste `N_SPLITS` na célula de configuração se quiser algo mais rápido.

**Saída:** `resultados/metricas_legalbert.json` + tabela comparativa com
baseline (02) e TextCNN (03), se os JSONs desses notebooks já existirem.


## 1. Setup


In [ ]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('repo em', REPO_DIR)


In [ ]:
%pip -q install "transformers>=4.44" "peft>=0.13" "datasets>=2.20" accelerate scikit-learn pyarrow scipy nltk
import torch
print('deps ok | torch', torch.__version__)


## 2. Correção — remove `torchao` incompatível com `peft`

O Colab pré-instala `torchao==0.10.0`, mas `peft>=0.13` exige `>=0.16.0`
quando o torchao está presente no ambiente. Sem este passo,
`aplicar_lora()` (via `get_peft_model`) falha com `ImportError`.
**Necessário rodar em toda sessão nova do Colab.**

In [ ]:
import importlib, subprocess

if importlib.util.find_spec('torchao') is not None:
    subprocess.run(['pip', 'uninstall', '-y', 'torchao'], check=True)
    print('torchao removido — OK')
else:
    print('torchao já ausente — OK')

## 3. Checagem de GPU


In [ ]:
import torch
USAR_GPU = torch.cuda.is_available()
if USAR_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: GPU não detectada.')
    print('Ative em Runtime > Alterar tipo de ambiente de execução > T4.')
    print('Sem GPU, a célula de configuração a seguir reduz o escopo para apenas'
          ' validar o fluxo (números NÃO são finais).')


## 4. Configuração

Com GPU: escopo completo (`n_splits=5`, `max_length=512`, `epochs=5`, `batch_size=16`),
espelhando os parâmetros testados em `CLAUDE.md`.

Sem GPU: reduzido automaticamente (`n_splits=3`, `max_length=128`, `epochs=1`,
`batch_size=8`) — só para checar que o código roda ponta a ponta.


In [ ]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

RANDOM_STATE = 42
LOSS = 'weighted_ce'    # 'weighted_ce' (padrão) ou 'focal' (desbalanceamento severo)
FOCAL_GAMMA = 2.0
USAR_LORA = True

if USAR_GPU:
    N_SPLITS, MAX_LENGTH, EPOCHS, BATCH_SIZE = 5, 512, 5, 16
    print('GPU detectada -> config completa (n_splits=5, max_length=512, epochs=5)')
else:
    N_SPLITS, MAX_LENGTH, EPOCHS, BATCH_SIZE = 3, 128, 1, 8
    print('SEM GPU -> config reduzida apenas para validar o fluxo'
          ' (n_splits=3, max_length=128, epochs=1).')

BASE = Path(REPO_DIR)

# Mesma lógica de persistência dos notebooks 01/02/03: prefere o Google Drive
# se já houver dados lá (gerados pelo 01), com fallback para o clone local.
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível (fora do Colab?): {_e}')

_drive_interim = (DRIVE_ROOT / 'data' / 'interim') if DRIVE_ROOT else None
PERSIST_BASE = DRIVE_ROOT if (_drive_interim and _drive_interim.exists()) else BASE

DATA_INTERIM = PERSIST_BASE / 'data' / 'interim'
RESULTADOS = PERSIST_BASE / 'resultados'
FIGURAS = RESULTADOS / 'figuras'; FIGURAS.mkdir(parents=True, exist_ok=True)

print(f'Lendo/gravando dados em: {PERSIST_BASE}')


## 5. Carrega o corpus (gerado pelo notebook 01)


In [ ]:
def _checar_arquivo(caminho, persist_base):
    if not caminho.exists():
        raise FileNotFoundError(
            f"Não encontrei {caminho}.\n"
            f"PERSIST_BASE resolvido para: {persist_base}\n"
            "Verifique: (1) o Google Drive foi montado com sucesso nesta sessão — "
            "rode `from google.colab import drive; drive.mount('/content/drive', "
            "force_remount=True)` e confira se "
            "'/content/drive/MyDrive/deep-acordao-tcu2/data/interim/' tem o parquet; "
            "(2) o notebook 01 já foi executado ao menos uma vez (local ou "
            "salvando no Drive)."
        )

import pandas as pd

_arq = DATA_INTERIM / 'acordaos_rotulados.parquet'
_checar_arquivo(_arq, PERSIST_BASE)
df = pd.read_parquet(_arq)
print('corpus n =', len(df))
print(df['LABEL'].value_counts().to_string())

contagens = df['LABEL'].value_counts()
classe_minima = contagens.min()
if classe_minima < N_SPLITS:
    print(f"AVISO: classe mais rara tem {classe_minima} amostras — reduzindo"
          f" N_SPLITS de {N_SPLITS} para {classe_minima}.")
    N_SPLITS = int(classe_minima)


## 6. Fine-tuning LoRA K-Fold — VOTO_LIMPO ponderado

`WeightedTrainer` (cross-entropy ponderada) por padrão. Use `LOSS='focal'` na
célula de configuração para a variante Focal Loss (Lin et al., 2017).


In [ ]:
from src.modelos.transformer import kfold

# Checkpoints ficam no disco LOCAL do Colab (não no Drive) — são artefatos
# transitórios de treino (early stopping); só o JSON final de métricas
# (célula de Persistência) precisa sobreviver à sessão.
CHECKPOINT_DIR = Path('/content/modelos_legalbert')

res_legalbert = kfold(
    df, campo='VOTO_LIMPO', n_splits=N_SPLITS,
    loss=LOSS, focal_gamma=FOCAL_GAMMA, usar_lora=USAR_LORA,
    epochs=EPOCHS, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
    output_base=str(CHECKPOINT_DIR),
)
print(f"F1-macro = {res_legalbert['mean_f1']:.4f}  IC95={res_legalbert['ci_95']}")
print(f"Acurácia média = {res_legalbert['mean_acc']:.4f}")


## 7. (Opcional) Focal Loss para contraste

Só execute esta célula se quiser comparar `weighted_ce` vs `focal` — **dobra o
tempo de GPU** (mais um K-Fold completo). Deixe comentado para pular.


In [ ]:
RODAR_FOCAL = False  # mude para True para rodar o contraste

res_legalbert_focal = None
if RODAR_FOCAL:
    res_legalbert_focal = kfold(
        df, campo='VOTO_LIMPO', n_splits=N_SPLITS,
        loss='focal', focal_gamma=FOCAL_GAMMA, usar_lora=USAR_LORA,
        epochs=EPOCHS, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
        output_base=str(Path('/content/modelos_legalbert_focal')),
    )
    print(f"F1-macro (focal) = {res_legalbert_focal['mean_f1']:.4f}"
          f"  IC95={res_legalbert_focal['ci_95']}")
else:
    print('RODAR_FOCAL=False — pulando contraste com Focal Loss.')


## 8. Comparação com baseline (02) e TextCNN (03)

Carrega os JSONs desses notebooks (se existirem no mesmo `PERSIST_BASE`) e monta
uma tabela única de F1-macro.


In [ ]:
import json
from src.avaliacao.metricas import comparar_modelos

todos_resultados = {'LegalBert-pt + LoRA (' + LOSS + ')': res_legalbert}
if res_legalbert_focal is not None:
    todos_resultados['LegalBert-pt + LoRA (focal)'] = res_legalbert_focal

caminho_baseline = RESULTADOS / 'metricas_baseline.json'
if caminho_baseline.exists():
    m = json.loads(caminho_baseline.read_text())
    todos_resultados['TF-IDF + LogReg (balanced)'] = m['kfold_5x_balanced']
else:
    print(f'Aviso: {caminho_baseline} não encontrado — rode o notebook 02 antes.')

caminho_textcnn = RESULTADOS / 'metricas_textcnn.json'
if caminho_textcnn.exists():
    m = json.loads(caminho_textcnn.read_text())
    todos_resultados['TextCNN'] = m['kfold_5x']
else:
    print(f'Aviso: {caminho_textcnn} não encontrado — rode o notebook 03 antes.')

tabela = comparar_modelos(todos_resultados)
print(tabela.to_string(index=False))
tabela.to_csv(RESULTADOS / 'comparacao_modelos.csv', index=False)


## 9. Persistência


In [ ]:
from src.avaliacao.metricas import salvar_json

saida = {
    'modelo': 'LegalBert-pt + LoRA',
    'feature': 'VOTO_LIMPO',
    'loss': LOSS,
    'usar_lora': USAR_LORA,
    'usar_gpu': USAR_GPU,
    'hiperparametros': {
        'n_splits': N_SPLITS, 'max_length': MAX_LENGTH,
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
    },
    'kfold': {
        'mean_f1': res_legalbert['mean_f1'],
        'std_f1': res_legalbert['std_f1'],
        'ci_95': list(res_legalbert['ci_95']),
        'mean_acc': res_legalbert['mean_acc'],
        'fold_scores': res_legalbert['fold_scores'],
    },
}
if res_legalbert_focal is not None:
    saida['kfold_focal'] = {
        'mean_f1': res_legalbert_focal['mean_f1'],
        'ci_95': list(res_legalbert_focal['ci_95']),
        'mean_acc': res_legalbert_focal['mean_acc'],
    }

salvar_json(saida, RESULTADOS / 'metricas_legalbert.json')
print(f'OK — persistido em: {PERSIST_BASE}')
if not USAR_GPU:
    print('LEMBRETE: esta execução rodou em modo reduzido (sem GPU) —'
          ' os números acima validam o fluxo, não são resultado final.')
